<a href="https://colab.research.google.com/github/thatcherty/ML-Algo-Selection/blob/main/ML_Algo_Selection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML Algo Selection

In [3]:
# Common
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import time

# requires pip install ucimlrepo
from ucimlrepo import fetch_ucirepo


from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neural_network import MLPClassifier, MLPRegressor
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import make_scorer, f1_score, precision_score, recall_score, accuracy_score


# additional for Neural Network
# include MLP or CNN
import tensorflow as tf
from tensorflow import keras


In [4]:
import argparse
from pydoc import text
from pyexpat import features
import re
from collections import OrderedDict
from datetime import datetime, timezone
from pathlib import Path
from typing import Iterable
from urllib.parse import parse_qsl, urlencode, urljoin, urlparse, urlunparse

import requests
from bs4 import BeautifulSoup
from openpyxl import Workbook
from openpyxl.styles import Alignment, Font, PatternFill

DEFAULT_URL = (
    "https://archive.ics.uci.edu/datasets"
    "?Task=Classification&Python=true&skip=0&take=10&sort=desc&orderBy=NumHits&search="
)
USER_AGENT = (
    "Mozilla/5.0 (X11; Linux x86_64) AppleWebKit/537.36 "
    "(KHTML, like Gecko) Chrome/123.0 Safari/537.36"
)


def build_page_url(base_url: str, *, skip: int, take: int) -> str:
    parsed = urlparse(base_url)
    params = OrderedDict(parse_qsl(parsed.query, keep_blank_values=True))
    params["skip"] = str(skip)
    params["take"] = str(take)
    new_query = urlencode(list(params.items()), doseq=True)
    return urlunparse(parsed._replace(query=new_query))


def extract_total_count(page_text: str) -> int | None:
    # Examples: "0 to 10 of 173" or "10 to 20 of 173"
    match = re.search(r"\b\d+\s+to\s+\d+\s+of\s+(\d+)\b", page_text)
    return int(match.group(1)) if match else None

def normalize_name(text: str) -> str:
    return re.sub(r"\s+", " ", text).strip()

def parse_dataset_links(html: str, page_url: str) -> list[int]:
    soup = BeautifulSoup(html, "html.parser")

    # Prefer title links that appear inside headings on the current UCI site.
    anchors = soup.select("h2 a[href]")

    # Fallback: any visible link pointing to a dataset page.
    if not anchors:
        anchors = soup.select("a[href*='/dataset/'], a[href*='/ml/datasets/']")

    dataset_id: list[int] = list()
    seen: set[tuple[str, str]] = set()

    for a in anchors:
        href = (a.get("href") or "").strip()
        name = normalize_name(a.get_text(" ", strip=True))

        if not href:
            continue
        if "/dataset/" not in href and "/ml/datasets/" not in href:
            continue
        if not name:
            continue

        full_url = urljoin(page_url, href)
        key = (name.casefold(), full_url)
        if key in seen:
            continue
        seen.add(key)
        # Extract the integer ID from the URL
        id_matches = re.findall(r'\d+', full_url)
        if id_matches:
            dataset_id.append(int(id_matches[0]))
    return dataset_id


def scrape_all_datasets(base_url: str, *, take: int = 25, timeout: int = 30) -> list[int]:
    session = requests.Session()
    session.headers.update({"User-Agent": USER_AGENT})

    all_ids: list[int] = list()
    seen_ids: set[int] = set()
    skip = 0
    total_count: int | None = None

    while True:
        page_url = build_page_url(base_url, skip=skip, take=take)
        response = session.get(page_url, timeout=timeout)
        response.raise_for_status()
        html = response.text
        text = BeautifulSoup(html, "html.parser").get_text(" ", strip=True)

        if total_count is None:
            total_count = extract_total_count(text)

        page_ids = parse_dataset_links(html, page_url)
        page_new_rows = [id for id in page_ids if id not in seen_ids]

        if not page_new_rows:
            break

        for id in page_new_rows:
            seen_ids.add(id)
            all_ids.append(id)

        if total_count is not None and len(all_ids) >= total_count:
            break

        if len(page_ids) < take:
            break

        skip += take

    return all_ids



def get_data() -> list[int]:
    parser = argparse.ArgumentParser(
        description="Scrape UCI dataset names + links from a filtered listing page into Excel."
    )
    parser.add_argument("--url", default=DEFAULT_URL, help="Filtered UCI listing URL to scrape.")
    parser.add_argument(
        "--take",
        type=int,
        default=25,
        help="Rows requested per page while paginating (default: 25).",
    )
    parser.add_argument(
        "--timeout",
        type=int,
        default=30,
        help="HTTP timeout in seconds (default: 30).",
    )
    args = parser.parse_args([])

    ids = scrape_all_datasets(args.url, take=args.take, timeout=args.timeout)
    if not ids:
        raise SystemExit("No dataset rows were found. The page structure may have changed.")
        return []

    return ids

In [5]:

# extract datasets
ids = get_data()
print(ids)


[53, 45, 186, 17, 222, 2, 320, 352, 109, 19, 697, 350, 144, 73, 544, 1, 891, 94, 468, 159, 15, 296, 14, 519, 602, 27, 20, 336, 242, 601, 292, 174, 327, 80, 545, 42, 267, 856, 31, 111, 555, 967, 62, 332, 597, 863, 373, 59, 529, 579, 936, 46, 942, 848, 563, 890, 52, 878, 383, 225, 572, 850, 887, 857, 445, 158, 151, 101, 571, 145, 938, 915, 484, 880, 864, 365, 110, 547, 759, 244, 193, 732, 763, 33, 105, 76, 143, 451, 176, 426, 603, 198, 12, 16, 264, 379, 471, 760, 39, 212, 461, 172, 591, 467, 90, 536, 50, 503, 47, 419, 161, 117, 81, 357, 485, 911, 312, 58, 342, 537, 827, 30, 329, 582, 799, 247, 40, 43, 26, 565, 380, 149, 146, 13, 38, 270, 372, 148, 913, 367, 229, 257, 277, 713, 300, 54, 28, 83, 728, 722, 22, 567, 184, 78, 107, 95, 63, 755, 3, 70, 44, 75, 69, 91, 23, 74, 32, 96, 82, 18, 88, 8, 147]


In [6]:
# Helper functions
def build_preprocessor(X: pd.DataFrame):
    # Detect column types
    numeric_cols = X.select_dtypes(include=[np.number]).columns.tolist()
    categorical_cols = [col for col in X.columns if col not in numeric_cols]

    # Numeric preprocessing
    numeric_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    # Categorical preprocessing
    categorical_pipeline = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OneHotEncoder(handle_unknown="ignore"))
    ])

    # Combine both
    preprocessor = ColumnTransformer([
        ("num", numeric_pipeline, numeric_cols),
        ("cat", categorical_pipeline, categorical_cols)
    ])

    return preprocessor

In [ ]:
# Load Datasets
count = 0

test_log = 1
test_nn = 1
test_tree = 1

# Create final dataset obj
algo_selection = pd.DataFrame({"Dataset": [], "Feature Count": [], "Instances": [], "Missing Values": [], "Recommended Algorithm": []})

for id in ids:
  if count == 25:
    break
  count += 1


  # load dataset
  dataset = fetch_ucirepo(id=id)
  print(id)

  # Address multiple target variables
  target_y = dataset.data.targets

  # if no target, do not include in evaluation
  if target_y is None or target_y.empty:
    print(f"No target variables for {id}")
    continue

  # if one target, simply use
  if len(target_y.columns) == 1:
    target_y = dataset.data.targets

  # if > 1 target, split to columns
  else:
    target_y = target_y.columns.tolist()

  # model assessment loop to account for multiple target var
  for target in target_y:
    X = dataset.data.features
    y = dataset.data.targets[target]

    # Get dataset features
    name = dataset.metadata['name']
    feature_count = dataset.metadata['num_features']
    instance_count = dataset.metadata['num_instances']
    missing_values = 'Yes' if 'Yes' in dataset.variables['missing_values'] else 'No'

    new_row = {'Dataset': name, 'Feature Count': feature_count, 'Instances': instance_count, 'Missing Values': missing_values}
    new_row_df = pd.DataFrame([new_row])

    # Add features to final dataset obj
    algo_selection = pd.concat([algo_selection, new_row_df], ignore_index=True)

    if test_log:
      # preprocessor will determine what preprocessing to perform on what columns
      preprocessor = build_preprocessor(X)

      # Logistic Regression
      logx_train, logx_test, logy_train, logy_test = train_test_split(X, y, test_size=0.2, random_state=0)

      # preprocess data and select DecisionTree
      pipe = Pipeline([('preprocessor', preprocessor), ('Logistic Regression', LogisticRegression())])
      pipe.fit(logx_train, logy_train)
      print(f"Logistic {id} accuracy is {pipe.score(logx_test, logy_test)}")

    if test_nn:
      # preprocessor will determine what preprocessing to perform on what columns
      preprocessor = build_preprocessor(X)

      # Neural Network
      nnx_train, nnx_test, nny_train, nny_test = train_test_split(X, y, test_size=0.2, random_state=0)

      # preprocess data and select DecisionTree
      pipe = Pipeline([('preprocessor', preprocessor), ('Logistic Regression', MLPClassifier(random_state=1, max_iter=500))])
      pipe.fit(nnx_train, nny_train)
      print(f"NN {id} accuracy is {pipe.score(nnx_test, nny_test)}")

    if test_tree:
      # Decision Tree

      # preprocessor will determine what preprocessing to perform on what columns
      preprocessor = build_preprocessor(X)

      decx_train, decx_test, decy_train, decy_test = train_test_split(X, y, test_size=0.2, random_state=0)

      # preprocess data and select DecisionTree
      pipe = Pipeline([('preprocessor', preprocessor), ('Tree', DecisionTreeClassifier())])
      pipe.fit(decx_train, decy_train)
      print(f"Tree {id} accuracy is {pipe.score(decx_test, decy_test)}")


    # Determine Recommended Algorithm

53
Logistic 53 accuracy is 1.0
NN 53 accuracy is 1.0
Tree 53 accuracy is 1.0
45
Logistic 45 accuracy is 0.6065573770491803


/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (500) reached and the optimization hasn't converged yet.
  warnings.warn(


NN 45 accuracy is 0.5901639344262295
Tree 45 accuracy is 0.4918032786885246
186
Logistic 186 accuracy is 0.53
NN 186 accuracy is 0.58
Tree 186 accuracy is 0.5907692307692308
17
Logistic 17 accuracy is 0.9649122807017544
NN 17 accuracy is 0.9649122807017544
Tree 17 accuracy is 0.9122807017543859
222
Logistic 222 accuracy is 0.8984850160345018
NN 222 accuracy is 0.891850049762247
Tree 222 accuracy is 0.8666371779276789
2


/usr/local/lib/python3.12/dist-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


Logistic 2 accuracy is 0.5784624833657488
